In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from tabulate import tabulate

from pkg.utils.analyze import summarize_outputs_in_df

pd.options.mode.chained_assignment = None

educat_path = "/home/knowledge-tracing/stedu/educat_baselines"
random_path = "/home/knowledge-tracing/stedu/random"
uncertainty_path = "/home/knowledge-tracing/stedu/uncertainty"
self_improvement_path = "/home/knowledge-tracing/stedu/self_improvement"

Load results

In [ ]:
# EduCAT
df_educat = summarize_outputs_in_df(outputs_dir=educat_path)

# Random and uncertainty
df_random = summarize_outputs_in_df(outputs_dir=random_path)
df_uncertainty = summarize_outputs_in_df(outputs_dir=uncertainty_path)

# Self-improvement
df_self_improvement = summarize_outputs_in_df(outputs_dir=self_improvement_path)
df_self_improvement = df_self_improvement[
    (df_self_improvement["force_random_num_given_positions"] == False)
]


### EduCAT

In [ ]:
EDUCAT_MODELS = {
    "random": "IRT w/ Random",
    "mfi": "IRT w/ MFI",
    "kli": "IRT w/ KLI",
    "becat": "IRT w/ BECAT",
}

df_results_educat = pd.DataFrame(
    data=zip(EDUCAT_MODELS.keys(), EDUCAT_MODELS.values()), columns=["Index", "Model"]
).set_index("Index")


def get_value(df: pd.DataFrame, strategy: str, seed: int, label: str) -> float:
    return df[(df["test_config.strategy"] == strategy) & (df["data_seed"] == seed)][
        label
    ].item()


for l in [5, 10, 20]:
    results = {}
    for strategy in EDUCAT_MODELS.keys():
        results[strategy] = {
            f"test_num_given_{l}_{metric}": [
                get_value(
                    df=df_educat,
                    strategy=strategy,
                    seed=i,
                    label=f"test_num_given_{l}_{metric}",
                )
                for i in range(5)
            ]
            for metric in ["acc", "auc"]
        }
    df_results_educat = df_results_educat.join(
        pd.DataFrame.from_dict(results, orient="index")
    )

### STEdu models

In [ ]:
def get_value(
    df: pd.DataFrame, num_given_positions: str, seed: int, label: str
) -> float:
    return df[
        (df["num_given_positions"] == num_given_positions) & (df["data.seed"] == seed)
    ][label].item()

Random

In [ ]:
RANDOM_MODELS = {"stedu_random": "Our w/ Random"}

df_results_random = pd.DataFrame(
    data=zip(RANDOM_MODELS.keys(), RANDOM_MODELS.values()), columns=["Index", "Model"]
).set_index("Index")


for l in [5, 10, 20]:
    results = {}
    for strategy in RANDOM_MODELS.keys():
        results[strategy] = {
            f"test_num_given_{l}_{metric}": [
                get_value(
                    df=df_random,
                    num_given_positions=l,
                    seed=i,
                    label=f"policy_test_epoch_{metric}",
                )
                for i in range(5)
            ]
            for metric in ["acc", "auc"]
        }
    df_results_random = df_results_random.join(
        pd.DataFrame.from_dict(results, orient="index")
    )

Uncertainty

In [ ]:
UNCERTAINTY_MODELS = {"stedu_uncertainty": "Our w/ Uncertainty"}

df_results_uncertainty = pd.DataFrame(
    data=zip(UNCERTAINTY_MODELS.keys(), UNCERTAINTY_MODELS.values()),
    columns=["Index", "Model"],
).set_index("Index")


for l in [5, 10, 20]:
    results = {}
    for strategy in UNCERTAINTY_MODELS.keys():
        results[strategy] = {
            f"test_num_given_{l}_{metric}": [
                get_value(
                    df=df_uncertainty,
                    num_given_positions=l,
                    seed=i,
                    label=f"policy_test_epoch_{metric}",
                )
                for i in range(5)
            ]
            for metric in ["acc", "auc"]
        }
    df_results_uncertainty = df_results_uncertainty.join(
        pd.DataFrame.from_dict(results, orient="index")
    )

Self-improvement

In [ ]:
SELF_IMPROVEMENT_MODELS = {"stedu_self_improvement": "Our w/ Self-Improv."}

df_auc_results_self_improvement = pd.DataFrame(
    data=zip(SELF_IMPROVEMENT_MODELS.keys(), SELF_IMPROVEMENT_MODELS.values()),
    columns=["Index", "Model"],
).set_index("Index")


for l in [5, 10, 20]:
    results = {}
    for strategy in SELF_IMPROVEMENT_MODELS.keys():
        results[strategy] = {
            f"test_num_given_{l}_{metric}": [
                get_value(
                    df=df_self_improvement,
                    num_given_positions=l,
                    seed=i,
                    label=f"policy_test_epoch_{metric}",
                )
                for i in range(5)
            ]
            for metric in ["acc", "auc"]
        }
    df_auc_results_self_improvement = df_auc_results_self_improvement.join(
        pd.DataFrame.from_dict(results, orient="index")
    )

Combine all result dfs

In [ ]:
df_results = pd.concat(
    [
        df_results_educat,
        df_results_random,
        df_results_uncertainty,
        df_auc_results_self_improvement,
    ]
)

Compute columns of results table

In [ ]:
for d in [df_results]:
    for v in [
        "test_num_given_5_acc",
        "test_num_given_5_auc",
        "test_num_given_10_acc",
        "test_num_given_10_auc",
        "test_num_given_20_acc",
        "test_num_given_20_auc",
    ]:
        f_mean = lambda x: np.array(x).mean() if len(x) > 0 else None
        f_std = lambda x: np.array(x).std() if len(x) > 0 else None
        _f_sig = lambda a, b: (
            (
                "$\circ$"
                if stats.ttest_rel(a, b).pvalue > 0.01
                else ("$\\ast$" if (f_mean(b) - f_mean(a) > 0) else "$\\bullet$")
            )
            if (a != b) #.all()  # if the same entry is compared
            else "$\\; \\: $"
        )
        f_sig = lambda x: (
            _f_sig(x, d.iloc[d.index == "stedu_self_improvement"][v].values[0])
            if (len(x) > 0)
            else ""
        )
        d[f"{v}_mean"] = d[v].apply(f_mean)
        d[f"{v}_std"] = d[v].apply(f_std)
        d[f"{v}_sig"] = d[v].apply(f_sig)

Format results to LaTeX

In [ ]:
df_acc = df_results[["Model"] + [c for c in df_results.columns if "acc" in c]]
df_auc = df_results[["Model"] + [c for c in df_results.columns if "auc" in c]]

In [ ]:
def print_latex(df: pd.DataFrame, columns: dict):
    f = lambda x, m, n: (
        ("\\textbf{" if (x == m) else "")
        + ("\\underline{" if (x == n) else "")
        + "{0:.4f}".format(x)
        + ("}" if (x == m) else "")
        + ("}" if (x == n) else "")
    )
    g = lambda x: "{0:.4f}".format(x)
    for dataset in columns.values():
        max_val, sec_max_val = df[f"{dataset}_mean"].nlargest(2)
        df[dataset] = df.apply(
            lambda x: (
                (tmp := f(x[f"{dataset}_mean"], max_val, sec_max_val))
                + " $\pm$ "
                + g(x[f"{dataset}_std"])
                + f" {x[f'{dataset}_sig']}"
                if pd.notnull(x[f"{dataset}_mean"])
                else "---"
            ),
            axis=1,
        )
    df_out = df[["Model"] + list((columns).values())]
    df_out["Model"] = df_out["Model"].apply(lambda x: f"\\textbf{{{x}}}")
    df_out = df_out.set_index("Model")

    headers = [f"\\textbf{{{d}}}" for d in (columns).keys()]
    print(
        tabulate(
            df_out,
            headers=["\\textbf{{Metric@Step}}"] + headers,
            tablefmt="latex_raw",
            colalign=["right"] + ["center"] * len((columns).values()),
        )
    )

In [ ]:
print_latex(
    df_acc,
    columns={
        "Accuracy@5": "test_num_given_5_acc",
        "Accuracy@10": "test_num_given_10_acc",
        "Accuracy@20": "test_num_given_20_acc",
    },
)

In [ ]:
print_latex(
    df_auc,
    columns={
        "AUC@5": "test_num_given_5_auc",
        "AUC@10": "test_num_given_10_auc",
        "AUC@20": "test_num_given_20_auc",
    },
)